In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

# 🔷 Load dataset
df = pd.read_csv('/content/drive/MyDrive/dataset.csv')

# 🔷 Use ALL improved features
X = df[['avg_pkt_size','std_dev','max_pkt_size','min_pkt_size',
        'pkt_count','avg_iat','iat_std','duration',
        'pkt_rate','size_iqr','burstiness']]

y = df['label']

# 🔷 Encode labels
le = LabelEncoder()
y = le.fit_transform(y)

# 🔷 Train-test split (STRATIFIED ✅)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 🔷 Handle class imbalance (IMPORTANT)
# Compute class weights
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# 🔷 Train XGBoost (TUNED)
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_train, y_train, sample_weight=sample_weights)

# 🔷 Predict
y_pred = xgb.predict(X_test)

# 🔷 Evaluation
print("\n🔷 XGBoost Results")
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n",
      classification_report(y_test, y_pred, zero_division=0))

# 🔷 Save model + encoder
joblib.dump(xgb, '/content/drive/MyDrive/xgb_model.pkl')
joblib.dump(le, '/content/drive/MyDrive/label_encoder.pkl')

print("\n✅ XGBoost model saved")


🔷 XGBoost Results
Accuracy: 0.8333333333333334

Confusion Matrix:
 [[ 1  0  0  0  0  0  0]
 [ 0  7  0  1  0  2  0]
 [ 0  0  3  0  0  0  0]
 [ 0  1  0 12  0  0  0]
 [ 0  0  0  3  2  0  0]
 [ 0  0  0  0  0 22  0]
 [ 0  2  0  0  1  0  3]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       0.70      0.70      0.70        10
           2       1.00      1.00      1.00         3
           3       0.75      0.92      0.83        13
           4       0.67      0.40      0.50         5
           5       0.92      1.00      0.96        22
           6       1.00      0.50      0.67         6

    accuracy                           0.83        60
   macro avg       0.86      0.79      0.81        60
weighted avg       0.84      0.83      0.82        60


✅ XGBoost model saved
